# Set up and Get data

In [1]:
!pip install numpy==1.26.4 -qq
!pip install pandas==2.1.4 -qq
!pip install google-cloud-firestore==2.23.0 -qq
!pip install google-cloud-bigquery==3.40.0 -qq
!pip install openpyxl==3.1.2 -qq
!pip install db-dtypes==1.5.0 -qq

In [2]:
import calendar
from datetime import date
import os
import hashlib
import numpy as np
import pandas as pd
from datetime import datetime
from zoneinfo import ZoneInfo
from google.cloud import firestore
from google.cloud import bigquery

In [3]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/lakehouse/default/Files/real-estate-project-445516-firebase-adminsdk-fbsvc-8d09bd7be7.json" 
db = firestore.Client()  # sẽ dùng GOOGLE_APPLICATION_CREDENTIALS env var

In [4]:
client = bigquery.Client.from_service_account_json("/lakehouse/default/Files/real-estate-project-445516-83dc50b692bc.json")

## Param

In [ ]:
def generate_month_query(table_full_name: str, tz_name: str = "Asia/Ho_Chi_Minh", ref_date: datetime | None = None) -> str:
    """
    Trả về query cho:
    - Tháng hiện tại
    - Nếu hôm nay là ngày 1 -> trả về tháng trước
    """
    tz = ZoneInfo(tz_name)
    now = ref_date.astimezone(tz) if ref_date else datetime.now(tz)

    year = now.year
    month = now.month

    # Nếu là ngày 1 -> lùi về tháng trước
    if now.day == 1:
        if month == 1:
            month = 12
            year -= 1
        else:
            month -= 1

    first_day_str = f"{year:04d}-{month:02d}-01T00:00:00"
    last_day_num = calendar.monthrange(year, month)[1]
    last_day_str = f"{year:04d}-{month:02d}-{last_day_num:02d}T00:00:00"

    query = f"""select *
                from `{table_full_name}`
                where (`Ngày đăng` <= '{first_day_str}' and `Ngày hết hạn` >= '{first_day_str}') or
                      (`Ngày đăng` >= '{first_day_str}' and `Ngày đăng` <= '{last_day_str}');"""
    return query

In [6]:
table = "real-estate-project-445516.real_estate_data.ads_data"
query = generate_month_query(table)

In [7]:
df = client.query(query).to_dataframe()

In [8]:
# df = pd.read_csv("/lakehouse/default/Files/ads_data_2025_11(1).csv")

In [9]:
# Lấy tháng hiện tại
year_month = datetime.now().strftime("%Y-%m")

# Lấy tháng trước
today = date.today()
if today.month == 1:
    prev_year = today.year - 1
    prev_month = 12
else:
    prev_year = today.year
    prev_month = today.month - 1
prev_year_month = f"{prev_year}-{prev_month:02d}"

In [10]:
# year_month = "2025-12"
# prev_year_month = "2025-11"

## Preprocessing

In [11]:
def safe_mean_round(s):
    values = [x for x in s if pd.notna(x)]
    return round(float(np.nanmean(values)), 2) if values else None

In [12]:
# bỏ những dòng không có giá
df1 = df.dropna(subset=['Khoảng giá']).copy()

In [15]:
group_cols = [ 
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án', 
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào' 
]

agg_df = (
    df1
    .groupby(group_cols, dropna=False)
    .agg(
        distinct_khoang_gia = ('Khoảng giá', lambda s: np.sort(s.unique()).tolist()),
        mean_unique_khoang_gia = ('Khoảng giá', lambda s: np.mean(np.unique(s)) if len(np.unique(s))>0 else np.nan),
        unique_ads = ('Mã tin', lambda s: s.unique().tolist()),
        n_unique_prices = ('Khoảng giá', lambda s: len(np.unique(s)))
    )
    .reset_index()
)

# nếu muốn chuyển mean sang triệu hoặc tỷ (ví dụ sang triệu VND):
agg_df['mean_unique_khoang_gia_million'] = agg_df['mean_unique_khoang_gia'] / 1e6
# hoặc sang tỷ:
agg_df['mean_unique_khoang_gia_billion'] = agg_df['mean_unique_khoang_gia'] / 1e9
# Tính giá mỗi m2
agg_df = agg_df[agg_df["Diện tích"] != 0]
agg_df["mean_unique_khoang_gia_million/m2"] = agg_df.apply(
    lambda row: row["mean_unique_khoang_gia_million"] / row["Diện tích"] if row["Diện tích"] not in [0, None] else None,
    axis=1
)
agg_df["mean_unique_khoang_gia_billion/m2"] = agg_df.apply(
    lambda row: row["mean_unique_khoang_gia_billion"] / row["Diện tích"] if row["Diện tích"] not in [0, None] else None,
    axis=1
)


In [16]:
agg_df.shape

(323712, 22)

In [15]:
# làm tròn chữ cái đầu tiên của tất cả các giá trị trong cột bds_type
agg_df['Loại BĐS'] = agg_df['Loại BĐS'].str.strip().str.capitalize()

# rename loại bds
mapping = {
    "Nhà biệt thự, liền kề": "Nhà biệt thự / Nhà liền kề",
    "Nhà mặt phố": "Nhà phố",
    "Nhà riêng": "Nhà ở",
    "Nhà trọ, phòng trọ": "Nhà trọ"
}
agg_df["Loại BĐS"] = agg_df["Loại BĐS"].replace(mapping)


# Check province, district

In [16]:
province_dict = {
    'An Giang': ['Thoại Sơn', 'Long Xuyên', 'Phú Tân', 'Tân Châu', 'Châu Phú', 'Chợ Mới', 'Châu Đốc', 'Tịnh Biên', 'Châu Thành', 'Tri Tôn'],
    'Bà Rịa Vũng Tàu': ['Bà Rịa', 'Châu Đức', 'Long Điền', 'Long Đất', 'Phú Mỹ', 'Vũng Tàu', 'Xuyên Mộc', 'Côn Đảo'],
    'Bình Dương': ['Bàu Bàng', 'Bắc Tân Uyên', 'Bến Cát', 'Dĩ An', 'Dầu Tiếng', 'Phú Giáo', 'Thuận An', 'Thủ Dầu Một', 'Tân Uyên'],
    'Bình Phước': ['Bình Long', 'Bù Gia Mập', 'Bù Đăng', 'Bù Đốp', 'Chơn Thành', 'Lộc Ninh', 'Phú Riềng', 'Đồng Phú', 'Đồng Xoài', 'Hớn Quản', 'Phước Long'],
    'Bình Thuận': ['Bắc Bình', 'Hàm Thuận Bắc', 'Hàm Thuận Nam', 'Hàm Tân', 'Phan Thiết', 'Đảo Phú Quý', 'Tuy Phong', 'La Gi', 'Tánh Linh', 'Đức Linh'],
    'Bình Định': ['Phù Cát', 'Quy Nhơn', 'Tuy Phước', 'Vân Canh', 'An Nhơn', 'Hoài Nhơn', 'Phù Mỹ', 'Tây Sơn'],
    'Bạc Liêu': ['Bạc Liêu', 'Hòa Bình', 'Giá Rai', 'Phước Long', 'Đông Hải', 'Hồng Dân'],
    'Bắc Giang': ['Bắc Giang', 'Hiệp Hòa', 'Lạng Giang', 'Lục Nam', 'Tân Yên', 'Việt Yên', 'Yên Dũng', 'Lục Ngạn', 'Yên Thế', 'Sơn Động'],
    'Bắc Kạn': ['Bắc Kạn', 'Chợ Mới', 'Chợ Đồn', 'Ba Bể'],
    'Bắc Ninh': ['Bắc Ninh', 'Gia Bình', 'Lương Tài', 'Quế Võ', 'Thuận Thành', 'Tiên Du', 'Từ Sơn', 'Yên Phong'],
    'Bến Tre': ['Bến Tre', 'Châu Thành', 'Ba Tri', 'Mỏ Cày Nam', 'Bình Đại', 'Thạnh Phú', 'Mỏ Cày Bắc', 'Chợ Lách'],
    'Cao Bằng': ['Cao Bằng', 'Trà Lĩnh'],
    'Cà Mau': ['Cà Mau', 'Cái Nước', 'Năm Căn', 'Trần Văn Thời', 'U Minh', 'Đầm Dơi', 'Phú Tân'],
    'Cần Thơ': ['Bình Thủy', 'Cái Răng', 'Ninh Kiều', 'Phong Điền', 'Ô Môn', 'Thốt Nốt', 'Thới Lai', 'Vĩnh Thạnh', 'Cờ Đỏ'],
    'Gia Lai': ['Ia Grai', 'Plei Ku', 'ChưPRông', 'KBang', 'Chư Păh', 'Chư Sê', 'Đăk Đoa', 'An Khê', 'Chư Pưh', 'Mang Yang', 'Krông Pa', 'Kông Chro', 'Ia Pa', 'Đăk Pơ'],
    'Hà Giang': ['Hà Giang', 'Bắc Quang', 'Hoàng Su Phì', 'Đồng Văn', 'Vị Xuyên'],
    'Hà Nam': ['Duy Tiên', 'Kim Bảng', 'Phủ Lý', 'Thanh Liêm', 'Bình Lục', 'Lý Nhân'],
    'Hà Nội': ['Ba Vì', 'Ba Đình', 'Bắc Từ Liêm', 'Chương Mỹ', 'Cầu Giấy', 'Gia Lâm', 'Hai Bà Trưng', 'Hoài Đức', 'Hoàn Kiếm', 'Hoàng Mai', 'Hà Đông', 'Long Biên', 'Mê Linh', 'Mỹ Đức', 'Nam Từ Liêm', 'Phú Xuyên', 'Phúc Thọ', 'Quốc Oai', 'Sóc Sơn', 'Sơn Tây', 'Thanh Oai', 'Thanh Trì', 'Thanh Xuân', 'Thường Tín', 'Thạch Thất', 'Tây Hồ', 'Đan Phượng', 'Đông Anh', 'Đống Đa', 'Ứng Hòa'],
    'Hà Tĩnh': ['Can Lộc', 'Hà Tĩnh', 'Kỳ Anh', 'Nghi Xuân', 'Cẩm Xuyên', 'Thạch Hà', 'Lộc Hà', 'Hương Sơn', 'Hương Khê', 'Hồng Lĩnh', 'Đức Thọ'],
    'Hòa Bình': ['Cao Phong', 'Hòa Bình', 'Kim Bôi', 'Lương Sơn', 'Lạc Thủy', 'Mai Châu', 'Tân Lạc', 'Yên Thủy', 'Đà Bắc', 'Lạc Sơn'],
    'Hưng Yên': ['Hưng Yên', 'Khoái Châu', 'Kim Động', 'Mỹ Hào', 'Phù Cừ', 'Tiên Lữ', 'Văn Giang', 'Văn Lâm', 'Yên Mỹ', 'Ân Thi'],
    'Hải Dương': ['Chí Linh', 'Gia Lộc', 'Hải Dương', 'Bình Giang', 'Cẩm Giàng', 'Kim Thành', 'Nam Sách', 'Thanh Hà', 'Thanh Miện', 'Tứ Kỳ', 'Kinh Môn', 'Ninh Giang'],
    'Hải Phòng': ['An Dương', 'An Lão', 'Cát Hải', 'Dương Kinh', 'Hải An', 'Hồng Bàng', 'Kiến An', 'Kiến Thụy', 'Lê Chân', 'Ngô Quyền', 'Thủy Nguyên', 'Vĩnh Bảo', 'Đồ Sơn', 'Tiên Lãng'],
    'Hậu Giang': ['Châu Thành', 'Ngã Bảy', 'Vị Thanh', 'Châu Thành A', 'Phụng Hiệp', 'Thị xã Long Mỹ', 'Vị Thủy'],
    'Hồ Chí Minh': ['Bình Chánh', 'Bình Thạnh', 'Bình Tân', 'Cần Giờ', 'Củ Chi', 'Gò Vấp', 'Hóc Môn', 'Nhà Bè', 'Phú Nhuận', 'Quận 1', 'Quận 10', 'Quận 11', 'Quận 12', 'Quận 2', 'Quận 3', 'Quận 4', 'Quận 5', 'Quận 6', 'Quận 7', 'Quận 8', 'Quận 9', 'Thủ Đức', 'Tân Bình', 'Tân Phú'],
    'Khánh Hòa': ['Cam Lâm', 'Cam Ranh', 'Diên Khánh', 'Khánh Vĩnh', 'Nha Trang', 'Ninh Hòa', 'Vạn Ninh', 'Trường Sa', 'Khánh Sơn'],
    'Kiên Giang': ['Châu Thành', 'Phú Quốc', 'Rạch Giá', 'Hà Tiên', 'Kiên Hải', 'An Biên', 'Kiên Lương', 'Hòn Đất', 'Giồng Riềng', 'Gò Quao', 'Giang Thành'],
    'Kon Tum': ['Kon Rẫy', 'Đăk Glei', 'Đăk Hà', 'Ngọc Hồi', 'Kon Plông', 'KonTum', 'Sa Thầy'],
    'Lai Châu': ['Tam Đường', 'Lai Châu', 'Phong Thổ'],
    'Long An': ['Bến Lức', 'Châu Thành', 'Cần Giuộc', 'Cần Đước', 'Thủ Thừa', 'Tân An', 'Tân Hưng', 'Tân Trụ', 'Đức Huệ', 'Đức Hòa', 'Kiến Tường', 'Mộc Hóa', 'Thạnh Hóa', 'Vĩnh Hưng', 'Tân Thạnh'],
    'Lào Cai': ['Sa Pa', 'Lào Cai', 'Bảo Thắng', 'Bát Xát'],
    'Lâm Đồng': ['Bảo Lâm', 'Bảo Lộc', 'Di Linh', 'Lâm Hà', 'Lạc Dương', 'Đam Rông', 'Đà Lạt', 'Đức Trọng', 'Đạ Huoai', 'Đơn Dương', 'Đạ Tẻh', 'Cát Tiên'],
    'Lạng Sơn': ['Cao Lộc', 'Lạng Sơn', 'Hữu Lũng', 'Bắc Sơn'],
    'Nam Định': ['Giao Thủy', 'Hải Hậu', 'Nam Định', 'Mỹ Lộc', 'Vụ Bản', 'Xuân Trường', 'Nghĩa Hưng', 'Ý Yên', 'Nam Trực', 'Trực Ninh'],
    'Nghệ An': ['Yên Thành', 'Vinh', 'Quỳnh Lưu', 'Cửa Lò', 'Diễn Châu', 'Nam Đàn', 'Nghĩa Đàn', 'Hoàng Mai', 'Đô Lương', 'Nghi Lộc', 'Thái Hòa', 'Hưng Nguyên', 'Anh Sơn'],
    'Ninh Bình': ['Ninh Bình', 'Gia Viễn', 'Hoa Lư', 'Nho Quan', 'Yên Mô', 'Tam Điệp', 'Yên Khánh', 'Kim Sơn'],
    'Ninh Thuận': ['Ninh Hải', 'Ninh Phước', 'Phan Rang - Tháp Chàm', 'Thuận Nam', 'Ninh Sơn', 'Bác Ái', 'Thuận Bắc'],
    'Phú Thọ': ['Cẩm Khê', 'Lâm Thao', 'Phú Thọ', 'Tam Nông', 'Thanh Ba', 'Thanh Sơn', 'Thanh Thủy', 'Việt Trì', 'Phù Ninh', 'Tân Sơn', 'Yên Lập', 'Đoan Hùng', 'Hạ Hòa'],
    'Phú Yên': ['Tuy Hòa', 'Đông Hòa', 'Sông Cầu', 'Phú Hòa', 'Sơn Hòa', 'Tuy An', 'Sông Hinh', 'Đồng Xuân'],
    'Quảng Bình': ['Quảng Ninh', 'Đồng Hới', 'Ba Đồn', 'Tuyên Hóa', 'Bố Trạch', 'Lệ Thủy', 'Quảng Trạch', 'Minh Hóa'],
    'Quảng Nam': ['Nông Sơn', 'Duy Xuyên', 'Hội An', 'Núi Thành', 'Thăng Bình', 'Điện Bàn', 'Đông Giang', 'Đại Lộc', 'Phú Ninh', 'Quế Sơn', 'Tam Kỳ', 'Bắc Trà My', 'Tiên Phước', 'Nam Giang'],
    'Quảng Ngãi': ['Lý Sơn', 'Quảng Ngãi', 'Sơn Tịnh', 'Tư Nghĩa', 'Bình Sơn', 'Mộ Đức', 'Đức Phổ', 'Ba Tơ', 'Nghĩa Hành'],
    'Quảng Ninh': ['Cẩm Phả', 'Hạ Long', 'Móng Cái', 'Quảng Yên', 'Vân Đồn', 'Hải Hà', 'Uông Bí', 'Đông Triều', 'Cô Tô', 'Đầm Hà', 'Tiên Yên'],
    'Quảng Trị': ['Gio Linh', 'Hải Lăng', 'Triệu Phong', 'Vĩnh Linh', 'Đông Hà', 'Cam Lộ', 'Quảng Trị'],
    'Sóc Trăng': ['Mỹ Tú', 'Trần Đề', 'Mỹ Xuyên', 'Sóc Trăng', 'Thạnh Trị', 'Ngã Năm', 'Kế Sách', 'Châu Thành', 'Cù Lao Dung', 'Long Phú'],
    'Sơn La': ['Mộc Châu', 'Bắc Yên', 'Sơn La', 'Vân Hồ', 'Phù Yên', 'Sốp Cộp'],
    'Thanh Hóa': ['Hoằng Hóa', 'Nga Sơn', 'Nghi Sơn', 'Ngọc Lặc', 'Nông Cống', 'Sầm Sơn', 'Thanh Hóa', 'Thạch Thành', 'Triệu Sơn', 'Đông Sơn', 'Bỉm Sơn', 'Hà Trung', 'Bá Thước', 'Như Thanh', 'Như Xuân', 'Quảng Xương', 'Thọ Xuân', 'Yên Định', 'Cẩm Thủy', 'Hậu Lộc', 'Lang Chánh', 'Vĩnh Lộc', 'Thiệu Hóa', 'Thường Xuân'],
    'Thái Bình': ['Hưng Hà', 'Kiến Xương', 'Thái Bình', 'Vũ Thư', 'Đông Hưng', 'Tiền Hải', 'Quỳnh Phụ', 'Thái Thuỵ'],
    'Thái Nguyên': ['Phổ Yên', 'Sông Công', 'Thái Nguyên', 'Phú Bình', 'Đại Từ', 'Phú Lương', 'Võ Nhai', 'Đồng Hỷ'],
    'Thừa Thiên Huế': ['Huế', 'Hương Thủy', 'Phong Điền', 'Phú Lộc', 'Phú Vang', 'Hương Trà', 'Quảng Điền', 'A Lưới'],
    'Tiền Giang': ['Châu Thành', 'Cái Bè', 'Gò Công', 'Gò Công Tây', 'Gò Công Đông', 'Mỹ Tho', 'Tân Phước', 'Cai Lậy', 'Chợ Gạo', 'Tân Phú Đông', 'Huyện Cai Lậy'],
    'Trà Vinh': ['Trà Vinh', 'Càng Long', 'Trà Cú', 'Tiểu Cần', 'Duyên Hải', 'Châu Thành', 'Cầu Ngang'],
    'Tuyên Quang': ['Tuyên Quang', 'Yên Sơn', 'Sơn Dương'],
    'Tây Ninh': ['Châu Thành', 'Dương Minh Châu', 'Gò Dầu', 'Hòa Thành', 'Trảng Bàng', 'Tân Biên', 'Tân Châu', 'Tây Ninh', 'Bến Cầu'],
    'Vĩnh Long': ['Bình Minh', 'Tam Bình', 'Bình Tân', 'Vĩnh Long', 'Long Hồ', 'Mang Thít', 'Vũng Liêm', 'Trà Ôn'],
    'Vĩnh Phúc': ['Bình Xuyên', 'Lập Thạch', 'Phúc Yên', 'Tam Dương', 'Tam Đảo', 'Vĩnh Yên', 'Vĩnh Tường', 'Yên Lạc', 'Sông Lô'],
    'Yên Bái': ['Yên Bái', 'Trấn Yên', 'Văn Yên', 'Yên Bình', 'Nghĩa Lộ', 'Mù Cang Chải', 'Lục Yên', 'Văn Chấn'],
    'Điện Biên': ['Mường Nhé', 'Điện Biên', 'Điện Biên Phủ'],
    'Đà Nẵng': ['Cẩm Lệ', 'Hòa Vang', 'Hải Châu', 'Liên Chiểu', 'Ngũ Hành Sơn', 'Sơn Trà', 'Thanh Khê'],
    'Đắk Lắk': ['Buôn Hồ', 'Buôn Ma Thuột', 'Buôn Đôn', 'Cư Kuin', "Cư M'gar", 'Krông Năng', 'Lăk', 'Ea Kar', "M'Đrăk", 'Krông Ana', 'Krông Buk', 'Krông Pắc', "Ea H'Leo", 'Ea Súp', 'Krông Bông'],
    'Đắk Nông': ['Dăk GLong', "Dăk R'Lấp", 'Dăk Song', 'Gia Nghĩa', 'Cư Jút', 'Dăk Mil', 'Krông Nô', 'Tuy Đức'],
    'Đồng Nai': ['Biên Hòa', 'Cẩm Mỹ', 'Long Khánh', 'Long Thành', 'Nhơn Trạch', 'Thống Nhất', 'Trảng Bom', 'Vĩnh Cửu', 'Xuân Lộc', 'Định Quán', 'Tân Phú'],
    'Đồng Tháp': ['Lai Vung', 'Cao Lãnh', 'Sa Đéc', 'Tháp Mười', 'Huyện Cao Lãnh', 'Lấp Vò', 'Châu Thành', 'Hồng Ngự', 'Tam Nông', 'Thanh Bình', 'Tân Hồng']
}

In [17]:
# province_dict_prod = {}

# missing_provinces = set(province_dict) - set(province_dict_prod)
# print("missing provinces: \n", missing_provinces)

# missing_districts = {}
# for province, districts in province_dict.items():
#     if province in province_dict_prod:
#         diff = set(districts) - set(province_dict_prod[province])
#         if diff:
#             missing_districts[province] = sorted(diff)
#     else:
#         # nếu tỉnh chưa tồn tại thì toàn bộ quận/huyện đều thiếu
#         missing_districts[province] = districts
# print("missing_districts: \n", missing_districts)

In [18]:
geo_df = df[['Tỉnh thành phố', 'Quận']].drop_duplicates().dropna()

# group tỉnh → list quận
province_districts = (
    geo_df    
    .groupby('Tỉnh thành phố')['Quận']
    .apply(list)
    .reset_index()
)
province_dict1 = dict(
    zip(province_districts["Tỉnh thành phố"], province_districts["Quận"])
)

In [19]:
missing_provinces = set(province_dict1) - set(province_dict)
print("missing provinces: \n", missing_provinces)

missing provinces: 
 set()


In [20]:
missing_districts = {}

for province, districts in province_dict1.items():
    if province in province_dict:
        diff = set(districts) - set(province_dict[province])
        if diff:
            missing_districts[province] = sorted(diff)
    else:
        # nếu tỉnh chưa tồn tại thì toàn bộ quận/huyện đều thiếu
        missing_districts[province] = districts

print("missing_districts: \n", missing_districts)

missing_districts: 
 {'Bến Tre': ['Giồng Trôm'], 'Gia Lai': ['AYun Pa'], 'Lạng Sơn': ['Đình Lập'], 'Sơn La': ['Sông Mã'], 'Yên Bái': ['Trạm Tấu']}


# Loại bỏ nhiễu

In [21]:
ban_df = agg_df[agg_df["Loại quảng cáo"] == "Bán"]
thue_df = agg_df[agg_df["Loại quảng cáo"] == "Cho thuê"]

In [22]:
price_col = 'mean_unique_khoang_gia_million/m2'
group_cols = ['Loại BĐS', 'Tỉnh thành phố', 'Quận']

# thu thập index các dòng cần drop
idx_to_drop = []
for _, g in ban_df.groupby(group_cols):
    n = len(g)
    # luôn bỏ ít nhất 1 dòng
    k = max(1, int(np.ceil(n*0.01)))
    if k > 0:
        # chọn k dòng có giá cao nhất trong nhóm
        idx_to_drop.extend(g.nlargest(k, price_col).index.tolist())

# dataframe sau khi đã loại bỏ 1% hàng top theo mỗi nhóm
ban_df = ban_df.drop(index=idx_to_drop).reset_index(drop=True)

In [23]:
price_col = 'mean_unique_khoang_gia_million'
group_cols = ['Loại BĐS', 'Tỉnh thành phố', 'Quận']

# thu thập index các dòng cần drop
idx_to_drop = []
for _, g in thue_df.groupby(group_cols):
    n = len(g)
    # luôn bỏ ít nhất 1 dòng
    k = max(1, int(np.ceil(n*0.01)))
    if k > 0:
        # chọn k dòng có giá cao nhất trong nhóm
        idx_to_drop.extend(g.nlargest(k, price_col).index.tolist())

# dataframe sau khi đã loại bỏ 1% hàng top theo mỗi nhóm
thue_df = thue_df.drop(index=idx_to_drop).reset_index(drop=True)

# Tính giá BĐS Bán

## Group by tới Quận 

In [24]:
# --- 2) group lại theo 4 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận']

grouped = (
    ban_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million_per_m2 = ('mean_unique_khoang_gia_million/m2', safe_mean_round),
        avg_price_billion_per_m2 = ('mean_unique_khoang_gia_billion/m2', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
    'Quận': 'district'
}
grouped_by_quan = grouped.rename(columns=rename_map)


In [25]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped_by_quan.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million_per_m2') is None or (isinstance(row.get('avg_price_million_per_m2'), float) and np.isnan(row.get('avg_price_million_per_m2'))) else float(row.get('avg_price_million_per_m2')),
        "avg_price_billion": None if row.get('avg_price_billion_per_m2') is None or (isinstance(row.get('avg_price_billion_per_m2'), float) and np.isnan(row.get('avg_price_billion_per_m2'))) else float(row.get('avg_price_billion_per_m2')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")


Pushed 1845 documents to Firestore collection 'price_data' in 5 batch(es).


## Group by tới Tỉnh thành 

In [26]:
# --- 2) group lại theo 3 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố']

grouped = (
    ban_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million_per_m2 = ('mean_unique_khoang_gia_million/m2', safe_mean_round),
        avg_price_billion_per_m2 = ('mean_unique_khoang_gia_billion/m2', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
}
grouped = grouped.rename(columns=rename_map)


In [27]:
# thêm cột Quận 
grouped['district'] = "All"

In [28]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million_per_m2') is None or (isinstance(row.get('avg_price_million_per_m2'), float) and np.isnan(row.get('avg_price_million_per_m2'))) else float(row.get('avg_price_million_per_m2')),
        "avg_price_billion": None if row.get('avg_price_billion_per_m2') is None or (isinstance(row.get('avg_price_billion_per_m2'), float) and np.isnan(row.get('avg_price_billion_per_m2'))) else float(row.get('avg_price_billion_per_m2')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")


Pushed 426 documents to Firestore collection 'price_data' in 2 batch(es).


# Tính giá BĐS cho thuê

## Group by tới Quận 

In [29]:
# --- 2) group lại theo 4 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận']

grouped = (
    thue_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million = ('mean_unique_khoang_gia_million', safe_mean_round),
        avg_price_billion = ('mean_unique_khoang_gia_billion', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
    'Quận': 'district'
}
grouped = grouped.rename(columns=rename_map)


In [30]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million') is None or (isinstance(row.get('avg_price_million'), float) and np.isnan(row.get('avg_price_million'))) else float(row.get('avg_price_million')),
        "avg_price_billion": None if row.get('avg_price_billion') is None or (isinstance(row.get('avg_price_billion'), float) and np.isnan(row.get('avg_price_billion'))) else float(row.get('avg_price_billion')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")


Pushed 845 documents to Firestore collection 'price_data' in 3 batch(es).


## Group by tới Tỉnh thành 

In [31]:
# --- 2) group lại theo 4 cột yêu cầu, lấy mean của mean_unique_khoang_gia
group_cols = ['Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố']

grouped = (
    thue_df
    .groupby(group_cols, dropna=False)
    .agg(
        avg_price_million = ('mean_unique_khoang_gia_million', safe_mean_round),
        avg_price_billion = ('mean_unique_khoang_gia_billion', safe_mean_round),
    )
    .reset_index()
)

# --- 3) thêm cột year_month hiện tại
grouped['year_month'] = year_month

# --- 4) rename columns
rename_map = {
    'Loại quảng cáo': 'price_type',
    'Loại BĐS': 'bds_type',
    'Tỉnh thành phố': 'province',
}
grouped = grouped.rename(columns=rename_map)


In [32]:
# thêm cột Quận 
grouped['district'] = "All"

In [33]:
collection_name = "price_data"

# Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
batch = db.batch()
batch_size = 400
written = 0
committed_batches = 0

for idx, row in grouped.iterrows():
    # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
    raw_id = f"{row.get('price_type','')}/{row.get('bds_type','')}/{row.get('province','')}/{row.get('district','')}/{row['year_month']}"
    doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

    doc_ref = db.collection(collection_name).document(doc_id)

    # payload: chuyển các giá trị numpy -> native python
    payload = {
        "price_type": None if pd.isna(row.get('price_type')) else str(row.get('price_type')),
        "bds_type": None if pd.isna(row.get('bds_type')) else str(row.get('bds_type')),
        "province": None if pd.isna(row.get('province')) else str(row.get('province')),
        "district": None if pd.isna(row.get('district')) else str(row.get('district')),
        "avg_price_million": None if row.get('avg_price_million') is None or (isinstance(row.get('avg_price_million'), float) and np.isnan(row.get('avg_price_million'))) else float(row.get('avg_price_million')),
        "avg_price_billion": None if row.get('avg_price_billion') is None or (isinstance(row.get('avg_price_billion'), float) and np.isnan(row.get('avg_price_billion'))) else float(row.get('avg_price_billion')),
        "year_month": row['year_month'],
        "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
    }

    # set (merge True để upsert - update nếu tồn tại)
    batch.set(doc_ref, payload, merge=True)
    written += 1

    if written % batch_size == 0:
        batch.commit()
        committed_batches += 1
        batch = db.batch()

# commit remaining
if written % batch_size != 0:
    batch.commit()
    committed_batches += 1

print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")

Pushed 223 documents to Firestore collection 'price_data' in 1 batch(es).


# Tính metric trên trang chủ

In [34]:
def push_to_firestore(grouped: pd.DataFrame, metric_name: str, collection_name = "metrics"):
    # Lấy giá tháng trước
    docs = (
        db.collection("metrics")
        .where("year_month", "==", prev_year_month)
        .where("name", "==", metric_name)
        .stream()
    )
    LM_values = [doc.to_dict() for doc in docs]
    LM_df = pd.DataFrame(LM_values)

    # thêm cột cho grouped
    grouped['year_month'] = year_month
    grouped['name'] = metric_name

    # rename columns
    rename_map = {
        'Tỉnh thành phố': 'province',
    }
    grouped = grouped.rename(columns=rename_map)

    # --- thêm dòng All ---
    if metric_name in ["project_count", "sales_bds_count", "total_sales_amt"]:
        all_row = pd.DataFrame([{
            "name": metric_name,
            "province": "All",
            "year_month": year_month,
            "value": grouped['value'].sum()
        }])
        grouped = pd.concat([grouped, all_row], ignore_index=True)
        grouped['value'] = grouped['value'].round(2)

    # Tính % so với LM
    if LM_df.empty or metric_name not in ["project_count", "sales_bds_count", "total_sales_amt", "CHCC_price"]:
        grouped['vs_LM'] = 0
    else:
        LM_merge = LM_df[['name', 'province', 'value']].rename(columns={'value': 'value_LM'})
        grouped = grouped.merge(LM_merge, on=["name", 'province'], how='left')
        grouped['vs_LM'] = np.where(
            grouped['value_LM'].isna(),
            0,
            np.where(
                grouped['value_LM'] == 0,
                0,
                (grouped['value'] - grouped['value_LM']) / grouped['value_LM'] * 100
            )
        )
        grouped['vs_LM'] = grouped['vs_LM'].round(2)

    # Chuẩn hóa trước khi đưa vào firestore
    def normalize_value(v):
        if v is None:
            return None
        # list / tuple
        if isinstance(v, (list, tuple)):
            normalized = []
            for item in v:
                # tuple/list 2 phần tử → dict
                if isinstance(item, (list, tuple)) and len(item) == 2:
                    normalized.append({
                        "district": normalize_value(item[0]),
                        "price": normalize_value(item[1])
                    })
                else:
                    normalized.append(normalize_value(item))
            return normalized
        # numpy array
        if isinstance(v, np.ndarray):
            return normalize_value(v.tolist())
        # numpy scalar
        if isinstance(v, np.generic):
            return v.item()
        # pandas NaN (chỉ cho scalar)
        if pd.isna(v):
            return None
        return v

    # Firestore batch limit = 500; dùng batch commit theo chunk (dùng 400 để an toàn)
    batch = db.batch()
    batch_size = 400
    written = 0
    committed_batches = 0

    for idx, row in grouped.iterrows():
        # tạo doc id từ hash để tránh ký tự đặc biệt và đảm bảo duy nhất theo key + month
        raw_id = f"{row.get('name','')}/{row.get('province','')}/{row['year_month']}"
        doc_id = hashlib.md5(raw_id.encode('utf-8')).hexdigest()

        doc_ref = db.collection(collection_name).document(doc_id)

        # payload: chuyển các giá trị numpy -> native python
        payload = {
            "name": None if pd.isna(row.get('name')) else str(row.get('name')),
            "province": None if pd.isna(row.get('province')) else str(row.get('province')),
            "year_month": row['year_month'],
            "value": normalize_value(row.get('value')),
            "vs_LM": normalize_value(row.get('vs_LM')),
            "updated_at": firestore.SERVER_TIMESTAMP  # timestamp server side
        }

        # set (merge True để upsert - update nếu tồn tại)
        batch.set(doc_ref, payload, merge=True)
        written += 1

        if written % batch_size == 0:
            batch.commit()
            committed_batches += 1
            batch = db.batch()

    # commit remaining
    if written % batch_size != 0:
        batch.commit()
        committed_batches += 1

    print(f"Pushed {written} documents to Firestore collection '{collection_name}' in {committed_batches} batch(es).")

## Tính số dự án


In [35]:
metric_name = "project_count"
grouped = (
    agg_df
        .groupby("Tỉnh thành phố")["Tên dự án"]
        .nunique()
        .reset_index(name="value")
)
push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 64 documents to Firestore collection 'metrics' in 1 batch(es).


## Tính số bất động sản bán

In [36]:
metric_name = "sales_bds_count"
grouped = ban_df.groupby("Tỉnh thành phố").size().reset_index(name="value")
push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 61 documents to Firestore collection 'metrics' in 1 batch(es).


## Tính tổng giá trị

In [37]:
metric_name = "total_sales_amt"
grouped = ban_df.groupby("Tỉnh thành phố")["mean_unique_khoang_gia_billion"].sum().reset_index(name="value")
push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 61 documents to Firestore collection 'metrics' in 1 batch(es).


## Trung bình giá CHCC

In [38]:
metric_name = "CHCC_price"

df_CHCC = ban_df[ban_df["Loại BĐS"] == "Căn hộ chung cư"]

grouped = (
    df_CHCC.groupby("Tỉnh thành phố")["mean_unique_khoang_gia_million"]
    .mean()
    .reset_index(name="value")
)

# mean dựa trên dataframe gốc, không dùng grouped
grouped.loc[len(grouped)] = ["All", df_CHCC["mean_unique_khoang_gia_million"].mean()]

push_to_firestore(grouped, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 36 documents to Firestore collection 'metrics' in 1 batch(es).


## Giá theo diện tích

In [39]:
def push_to_firestore1(data, metric_name):
    # ================== CLEAN DATA ==================
    data = data.dropna(subset=['Diện tích', 'mean_unique_khoang_gia_million/m2'])

    # ================== BINS & LABELS ==================
    match metric_name: 
        case "CHCC_price_by_m2":
            bins = [0, 40, 60, 80, 100, 120, 140, 160, 180, 200, 220, 240]
        case "land_price_by_m2":
            bins = (
                [0]
                + list(range(40, 501, 20)) 
                + list(range(600, 3001, 100))
                + list(range(3500, 8001, 500))
            )
        case "nha_o_price_by_m2":
            bins = (
                [0, 30]
                + list(range(60, 401, 20)) 
            )

    labels = []
    for i in range(len(bins) - 1):
        left = bins[i]
        right = bins[i + 1]
        if left == 0:
            labels.append(f"<{right}")
        else:
            labels.append(f"{left}-{right}")

    data['khoảng diện tích'] = pd.cut(
        data['Diện tích'],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    # ================== GROUP BY ==================
    grouped1 = (
        data
        .groupby(['Tỉnh thành phố', 'khoảng diện tích'], observed=True)
        ['mean_unique_khoang_gia_million/m2']
        .mean()
        .reset_index()
        .rename(columns={
            'mean_unique_khoang_gia_million/m2': 'avg_price_million_per_m2'
        })
    )

    # ================== ENSURE FULL LABELS ==================
    provinces = grouped1['Tỉnh thành phố'].unique()

    full_index = pd.MultiIndex.from_product(
        [provinces, labels],
        names=['Tỉnh thành phố', 'khoảng diện tích']
    )

    grouped2 = (
        grouped1
        .set_index(['Tỉnh thành phố', 'khoảng diện tích'])
        .reindex(full_index)
        .reset_index()
    )

    # ================== FILL MISSING ==================
    grouped2['avg_price_million_per_m2'] = (
        grouped2['avg_price_million_per_m2']
        .fillna(-1)
        .round(2)
    )

    # ================== SORT ==================
    grouped2 = grouped2.sort_values(
        ['Tỉnh thành phố', 'khoảng diện tích']
    ).reset_index(drop=True)

    # === TÍNH CHO ALL (TOÀN BỘ DỮ LIỆU) ===
    rows_all = (
        data
        .groupby(['khoảng diện tích'], observed=True)['mean_unique_khoang_gia_million/m2']
        .mean()
        .reset_index()
        .assign(**{'Tỉnh thành phố': 'All'})
        .rename(columns={'mean_unique_khoang_gia_million/m2': 'avg_price_million_per_m2'})
    )

    # Làm tròn
    rows_all['avg_price_million_per_m2'] = rows_all['avg_price_million_per_m2'].round(2)

    # Đưa thứ tự cột giống result
    rows_all = rows_all[['Tỉnh thành phố', 'khoảng diện tích', 'avg_price_million_per_m2']]

    # === GHÉP VỚI KẾT QUẢ GỐC ===
    grouped2 = (
        pd.concat([grouped2, rows_all], ignore_index=True)
        .sort_values(['Tỉnh thành phố', 'khoảng diện tích'])
        .reset_index(drop=True)
    )
    grouped2["khoảng diện tích"] = pd.Categorical(
        grouped2["khoảng diện tích"],
        categories=labels,
        ordered=True
    )
    grouped_final = (
        grouped2
        .sort_values("khoảng diện tích")
        .groupby("Tỉnh thành phố", as_index=False)
        .agg({"avg_price_million_per_m2": list})
        .rename(columns={"avg_price_million_per_m2": "value"})
    )
    push_to_firestore(grouped_final, metric_name)



### Căn hộ chung cư

In [40]:
metric_name = "CHCC_price_by_m2"
data = ban_df[(ban_df["Loại BĐS"] == "Căn hộ chung cư") & (ban_df['Diện tích'] <= 240)]
push_to_firestore1(data, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 36 documents to Firestore collection 'metrics' in 1 batch(es).


### Đất

In [41]:
metric_name = "land_price_by_m2"
data = ban_df[(ban_df["Loại BĐS"] == "Đất") & (ban_df['Diện tích'] <= 8000)]
push_to_firestore1(data, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 58 documents to Firestore collection 'metrics' in 1 batch(es).


### Nhà ở

In [42]:
metric_name = "nha_o_price_by_m2"
data = ban_df[(ban_df["Loại BĐS"] == "Nhà ở") & (ban_df['Diện tích'] <= 400)]
push_to_firestore1(data, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 51 documents to Firestore collection 'metrics' in 1 batch(es).


## Top 5 quận huyện

In [43]:
def top5_quan(loai_bds):
    # Top 5 quận giá cao nhất mỗi tỉnh (Căn hộ chung cư)
    top5 = (
        grouped_by_quan[grouped_by_quan['bds_type'] == loai_bds]
        .groupby('province', group_keys=False)
        .apply(lambda x: x.nlargest(5, 'avg_price_million_per_m2'))
    )

    # Mỗi tỉnh -> 1 dòng, value là list[(district, price)]
    province_df = (
        top5
        .groupby('province')
        .apply(lambda x: list(zip(x['district'], x['avg_price_million_per_m2'])))
        .reset_index(name='value')
    )

    # Thêm dòng All (toàn quốc)
    final_df = pd.concat([
        province_df,
        pd.DataFrame({
            'province': ['All'],
            'value': [list(zip(
                top5.nlargest(5, 'avg_price_million_per_m2')['district'],
                top5.nlargest(5, 'avg_price_million_per_m2')['avg_price_million_per_m2']
            ))]
        })
    ], ignore_index=True)

    return final_df

### Nhà riêng

In [44]:
metric_name = "top 5 quan by nha rieng price"
top5_df = top5_quan("Nhà ở")
push_to_firestore(top5_df, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 51 documents to Firestore collection 'metrics' in 1 batch(es).


### Nhà phố

In [45]:
metric_name = "top 5 quan by nha pho price"
top5_df = top5_quan("Nhà phố")
push_to_firestore(top5_df, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 51 documents to Firestore collection 'metrics' in 1 batch(es).


### CHCC

In [46]:
metric_name = "top 5 quan by CHCC price"
top5_df = top5_quan("Căn hộ chung cư")
push_to_firestore(top5_df, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 36 documents to Firestore collection 'metrics' in 1 batch(es).


### Đất

In [47]:
metric_name = "top 5 quan by land price"
top5_df = top5_quan("Đất")
push_to_firestore(top5_df, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 58 documents to Firestore collection 'metrics' in 1 batch(es).


## Phân bố giá CHCC, nhà ở

In [48]:
def push_to_firestore2(data, metric_name):
    # ================== CLEAN DATA ==================
    # Drop rows missing area or price if area needed for conversion earlier; otherwise drop price NaNs
    data = data.dropna(subset=['mean_unique_khoang_gia'])

    # ================== PRICE BINS & LABELS ==================
    bins = [0, 3e9, 5e9, 7e9, 10e9, 15e9, 20e9, np.inf]
    labels = ["<3 tỷ", "3-5 tỷ", "5-7 tỷ", "7-10 tỷ", "10-15 tỷ", "15-20 tỷ", ">20 tỷ"]

    # Create categorical price range column
    data['khoảng giá'] = pd.cut(
        pd.to_numeric(data['mean_unique_khoang_gia'], errors='coerce'),
        bins=bins,
        labels=labels,
        include_lowest=True,
        right=False  # intervals: [left, right)
    )

    # ================== GROUP BY (COUNT ROWS) ==================
    grouped1 = (
        data
        .groupby(['Tỉnh thành phố', 'khoảng giá'], observed=True)
        .size()
        .reset_index(name='count')
    )

    # ================== ENSURE FULL LABELS FOR EVERY PROVINCE ==================
    provinces = grouped1['Tỉnh thành phố'].unique()
    full_index = pd.MultiIndex.from_product(
        [provinces, labels],
        names=['Tỉnh thành phố', 'khoảng giá']
    )

    grouped2 = (
        grouped1
        .set_index(['Tỉnh thành phố', 'khoảng giá'])
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    # ================== SORT ==================
    grouped2 = grouped2.sort_values(['Tỉnh thành phố', 'khoảng giá']).reset_index(drop=True)

    # === TÍNH CHO ALL (TOÀN BỘ DỮ LIỆU) ===
    rows_all = (
        data
        .groupby(['khoảng giá'], observed=True)
        .size()
        .reset_index(name='count')
        .assign(**{'Tỉnh thành phố': 'All'})
    )

    # Đảm bảo tất cả nhãn cũng có trong rows_all
    full_all_index = pd.MultiIndex.from_product([['All'], labels], names=['Tỉnh thành phố', 'khoảng giá'])
    rows_all = (
        rows_all
        .set_index(['Tỉnh thành phố','khoảng giá'])
        .reindex(full_all_index, fill_value=0)
        .reset_index()
    )

    # === GHÉP VỚI KẾT QUẢ GỐC ===
    grouped2 = (
        pd.concat([grouped2, rows_all], ignore_index=True)
        .sort_values(['Tỉnh thành phố', 'khoảng giá'])
        .reset_index(drop=True)
    )

    # Make sure 'khoảng giá' is categorical ordered as our labels
    grouped2["khoảng giá"] = pd.Categorical(
        grouped2["khoảng giá"],
        categories=labels,
        ordered=True
    )

    # Final shape: one row per province, column 'value' is list of counts in order of labels
    grouped_final = (
        grouped2
        .sort_values("khoảng giá")
        .groupby("Tỉnh thành phố", as_index=False)
        .agg({"count": list})
        .rename(columns={"count": "value"})
    )

    # Push to firestore (giữ nguyên gọi hàm)
    push_to_firestore(grouped_final, metric_name)


In [49]:
metric_name = "CHCC_count_by_price"
data = ban_df[(ban_df["Loại BĐS"] == "Căn hộ chung cư") & (ban_df['Diện tích'] <= 240)]
push_to_firestore2(data, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 36 documents to Firestore collection 'metrics' in 1 batch(es).


In [50]:
metric_name = "nha_o_count_by_price"
data = ban_df[(ban_df["Loại BĐS"] == "Nhà ở") & (ban_df['Diện tích'] <= 400)]
push_to_firestore2(data, metric_name)

/home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/google/cloud/firestore_v1/base_collection.py:316: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)
/tmp/ipykernel_62/18393674.py:6: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  .where("name", "==", metric_name)


Pushed 51 documents to Firestore collection 'metrics' in 1 batch(es).


# Xóa doc, count doc

In [51]:
# !pip install google-cloud-firestore -qq 
# import time
# from google.cloud import firestore
# from google.api_core import exceptions

# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/lakehouse/default/Files/real-estate-project-445516-firebase-adminsdk-fbsvc-8d09bd7be7.json" 
# db = firestore.Client()  # sẽ dùng GOOGLE_APPLICATION_CREDENTIALS env var

# def delete_collection(collection_path: str, batch_size: int = 500, max_retries: int = 5, sleep_between_batches: float = 0.1):
#     coll_ref = db.collection(collection_path)
#     # coll_ref = db.collection(collection_path).where("year_month", "==", "2025-12")

#     while True:
#         # Lấy 1 page
#         attempt = 0
#         docs = None
#         while attempt <= max_retries:
#             try:
#                 docs = coll_ref.limit(batch_size).get()   # tránh .stream()
#                 break
#             except (exceptions.DeadlineExceeded, exceptions.ServiceUnavailable, exceptions.InternalServerError) as e:
#                 attempt += 1
#                 wait = 2 ** attempt
#                 print(f"Get attempt {attempt} failed: {e}. retrying in {wait}s...")
#                 time.sleep(wait)
#         if docs is None:
#             raise RuntimeError("Failed to fetch documents after retries.")

#         if not docs:
#             print("No more documents — finished.")
#             return

#         # Xóa theo batch
#         batch = db.batch()
#         for d in docs:
#             batch.delete(d.reference)

#         attempt = 0
#         while attempt <= max_retries:
#             try:
#                 batch.commit()
#                 print(f"Deleted batch of {len(docs)} documents.")
#                 break
#             except (exceptions.DeadlineExceeded, exceptions.ServiceUnavailable, exceptions.InternalServerError) as e:
#                 attempt += 1
#                 wait = 2 ** attempt
#                 print(f"Commit attempt {attempt} failed: {e}. retrying in {wait}s...")
#                 time.sleep(wait)
#         else:
#             raise RuntimeError("Failed to commit delete batch after retries.")

#         time.sleep(sleep_between_batches)

# delete_collection("metrics")

In [52]:
# def count_documents():
#     coll_ref = db.collection("price_data")
#     aggregation_query = coll_ref.count()

#     result = aggregation_query.get()
#     print("Total documents:", result[0])

# count_documents()